<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase9-portfolio-launch/01_code_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 9: Lesson 1: Chroma Migration
# Phase 9: Code Optimization
## Section 1: Chroma Migration
**Goal**: Replace the Phase 7 dictionary-based knowledge base and
      keyword-overlap retrieval with a real vector database
      (Chroma) using semantic similarity search powered by
      Gemini embeddings.
**Regulatory mapping**: NIST AI 600-1 hallucination tracking,
                    source data lineage verification.
**Date**: June 2026.
**Status**: In Progress.

In [1]:
# Phase 9, Lesson 1: Upgrading the RAG Knowledge Base to Chroma
#
# Goal: Replace the Phase 7 Python dictionary knowledge base and keyword-overlap
#       retrieve_documents() function with a real vector database (Chroma) using
#       semantic similarity search powered by Gemini embeddings.
#
# What changes from Phase 7:
#    - KNOWLEDGE_BASE (dict) -> a Chroma collection
#    - retrieve_documents() keyword scoring -> Chroma's semantic query()
#    - rag_query return shape is UNCHANGED so the rest of the pipeline
#     (Observer Agent, Phase 8 orchestration) keeps working without
#     modification downstream.
#
# Regulatory mapping: NIST AI 600-1 hallucination tracking and source data
#                     lineage verification (unchanged from phase 7, now backed
#                     by a production-grade retrieval engine instead of a
#                     notebook-only keyword matcher).

!pip install chromadb

import time
import pandas as pd
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from google import genai
from google.genai import types
from google.colab import userdata, drive
import os

# Setup (Unchanged from Phase 7)
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))


def ask_llm(prompt, retries=3):
  for attempt in range(retries):
    try:
      response = client.models.generate_content(
          model="gemini-flash-latest",
          contents=prompt
      )
      return response.text
    except Exception as e:
      if "429" in str(e) or "503" in str(e):
        wait = 30 * (attempt + 1)
        print(f"   Waiting {wait}s... (attempt {attempt + 1}/{retries})")
        time.sleep(wait)
      else:
        raise e
  return "Error: max retries exceeded"

# =========== GEMINI EMBEDDING FUNCTION FOR CHROMA ==========
# Chroma needs an embedding function it can call automatically whenever
# documents are added or queried. This wraps the current Gemini
# embedding model (gemini-embedding-001) so Chroma can use it natively.
#
# document_mode controls task_type: documents being STORED use RETRIEVAL_DOCUMENT,
# queries being SEARCHED use RETRIEVAL_QUERY. These produce different embeddings
# optimised for each role, which is the single biggest accuracy improvement over
# Phase 7's keyword scoring, Phase 7 had no concept of "document vs query" framing at all.

class GeminiEmbeddingFunction(EmbeddingFunction):
  document_mode = True

  def __call__(self, input: Documents) -> Embeddings:
    embedding_task = "RETRIEVAL_DOCUMENT" if self.document_mode else "RETRIEVAL_QUERY"
    retries = 3
    for attempt in range(retries):
      try:
        response = client.models.embed_content(
            model="gemini-embedding-001",
            contents=input,
            config=types.EmbedContentConfig(task_type=embedding_task),
        )
        return [e.values for e in response.embeddings]
      except Exception as e:
        print(f"   DEBUG embedding error: {type(e).__name__}: {e}")  # <-- add this line
        if "429" in str(e) or "503" in str(e):
          wait = 30 * (attempt + 1)
          print(f"   Embedding retry, waiting {wait}s...")
          time.sleep(wait)
        else:
          raise e
    raise RuntimeError("Embedding failed after max retries")

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

# =========== SOURCE DOCUMENTS (same as in Phase 7)===========
SOURCE_DOCS = {
    "doc_001": {
        "title": "EU AI Act Article 5 - Prohibited Practices",
        "content": """Article 5 of the EU AI Act prohibits the following
AI practices: subliminal manipulation below the threshold of consciousness,
exploitation of vulnerabilities of specific groups, social scoring by public
authorities, real-time remote biometric identification in public spaces by
law enforcement except in narrowly defined cases, retrospective biometric
identification systems, emotion recognition in workplace and educational
settings, and biometric categorisation to infer sensitive characteristics
such as race, political opinions, or sexual orientation.""",
        "source": "EU AI Act 2024/1689, Article 5",
        "category": "regulation",
    },
    "doc_002": {
        "title": "EU AI Act Article 10 - Data Governance",
        "content": """Article 10 requires that training, validation and
testing datasets for high-risk AI systems must be relevant, representative,
free of error and complete. They must be examined for possible biases that
could lead to risks to fundamental rights or discrimination. When processing
special categories of personal data, appropriate safeguards must be in place.
Providers must implement data governance practices covering the intended
purpose, data collection processes, and examination for biases.""",
        "source": "EU AI Act 2024/1689, Article 10",
        "category": "regulation",
    },
    "doc_003": {
        "title": "EU AI Act Article 14 - Human Oversight",
        "content": """Article 14 requires that high-risk AI systems be
designed to allow effective oversight by natural persons during the period
of use. The system must enable those responsible for oversight to understand
the capabilities and limitations of the system, detect and address
malfunctions, and intervene or interrupt the system through a stop button
or similar procedure. Human oversight must be proportionate to the risks.""",
        "source": "EU AI Act 2024/1689, Article 14",
        "category": "regulation",
    },
    "doc_004": {
        "title": "EU AI Act Article 99 - Penalties",
        "content": """Article 99 establishes three penalty tiers.
Violations of prohibited practices under Article 5 carry fines of up to
35 million euros or 7 percent of global annual turnover. Violations of
obligations for high-risk AI systems under Chapter III and V carry fines
of up to 15 million euros or 3 percent of global annual turnover under
Article 99(3). Providing incorrect or misleading information to authorities
carries fines of up to 7.5 million euros or 1 percent of global annual
turnover.""",
        "source": "EU AI Act 2024/1689, Article 99",
        "category": "regulation",
    },
    "doc_005": {
        "title": "NIST AI RMF - Four Core Functions",
        "content": """The NIST AI Risk Management Framework organises
risk management into four core functions. GOVERN establishes organisational
policies, culture, and accountability for AI risk. MAP identifies the
context and specific risks of each AI application. MEASURE uses
quantitative and qualitative methods to analyse and assess AI risks.
MANAGE prioritises and acts on identified risks through controls,
monitoring, and incident response. These functions are intended to be
applied continuously across the AI lifecycle.""",
        "source": "NIST AI RMF 1.0, Core Functions",
        "category": "framework",
    },
    "doc_006": {
        "title": "NIST AI 600-1 - Generative AI Risk",
        "content": """NIST AI 600-1 addresses risks specific to generative
AI systems including large language models. Key risks include hallucination
where models produce plausible but false information, data memorisation
where models reproduce training data, harmful bias in generated content,
and confabulation of sources and citations. Organisations are required to
implement hallucination tracking, source data lineage verification, and
output monitoring to address these risks in production deployments.""",
        "source": "NIST AI 600-1, Generative AI Profile",
        "category": "framework",
    },
}

# ========== BUILD THE CHROMA COLLECTION ==========
# get_or_create_collection means re-running this cell is safe, it will not
# duplicate documents on a second run within the same session.

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="ai_governance_kb",
    embedding_function=embed_fn,
)

embed_fn.document_mode = True
collection.add(
    ids=list(SOURCE_DOCS.keys()),
    documents=[doc["content"] for doc in SOURCE_DOCS.values()],
    metadatas=[
        {"title": doc["title"], "source": doc["source"], "category": doc["category"]}
        for doc in SOURCE_DOCS.values()
    ],
)

print("====== CHROMA KNOWLEDGE BASE LOADED ======")
for doc_id, doc in SOURCE_DOCS.items():
  print(f"   {doc_id}: {doc['title']}")
print(f"\nTotal documents in collection: {collection.count()}")
print("\nChroma knowledge base ready (semantic retrieval, gemini-embedding-001) ✅")

# ========== RETRIEVAL ENGINE (SEMANTIC, REPLACES KEYWORD SCORING) ==========
# Chroma's query() does cosine similarity over the
# Gemini embeddings, which is what actually fixes the retrieval quality
# limitation documented in the Phase 7 findings (Article 5 vs Article 99
# being confused by surface keyword overlap on the phrase "AI systems").
def retrieve_documents(query, top_k=2):
  """Retrieve the most relevant documents for a query using semantic
  similarity search instead of keyword overlap."""
  embed_fn.document_mode = False  # switch to RETRIEVAL_QUERY mode for the query itself
  results = collection.query(
      query_texts=[query],
      n_results=top_k,
  )
  embed_fn.document_mode = True  # switch back for any subsequent document adds

  retrieved = []
  if results["documents"] and results["documents"][0]:
    for content, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
      retrieved.append({
          "title": metadata["title"],
          "content": content,
          "source": metadata["source"],
          "category": metadata["category"],
          "similarity_distance": distance,
      })
  return retrieved

# ============== RAG QUERY FUNCTION ==============
# Return shape is IDENTICAL to Phase 7's rag_query(): query, answer,
# sources, titles, grounded, doc_count. This is deliberate. The Observer
# Agent and Phase 8 orchestration consume this shape and should not
# need any changes downstream of this upgrade.

def rag_query(query, top_k=2):
  """Query the knowledge base with RAG grounding, now backed by
  semantic retrieval instead of keyword matching."""

  retrieved_docs = retrieve_documents(query, top_k)

  if not retrieved_docs:
    return {
        "query": query,
        "answer": "No relevant documents found.",
        "sources": [],
        "grounded": False,
        "doc_count": 0,
    }

  context = "\n\n".join([
      f"SOURCE: {doc['source']}\n{doc['content']}"
      for doc in retrieved_docs
  ])

  grounded_prompt = f"""You are an AI governance assistant.
Answer the question using ONLY the provided source documents.
Do not use any knowledge outside of these documents.
If the documents do not contain the answer, say so explicitly.

SOURCE DOCUMENTS:
{context}

QUESTION: {query}

ANSWER (based only on the source documents above):"""

  answer = ask_llm(grounded_prompt)

  return {
      "query": query,
      "answer": answer,
      "sources": [doc["source"] for doc in retrieved_docs],
      "titles": [doc["title"] for doc in retrieved_docs],
      "grounded": True,
      "doc_count": len(retrieved_docs),
  }

# ============== TEST THE UPGRADED RAG PIPELINE ==============
# Same test queries as Phase 7, including the exact query that exposed
# the keyword-retrieval failure (Article 5 incorrectly retrieved for a
# high-risk fine question). This is the direct before/after comparison.

test_queries = [
    "What does the EU AI Act say about prohibited AI practices?",
    "What are the penalty tiers under the EU AI Act?",
    "What is the NIST AI RMF and what are its four functions?",
    "What does Article 14 require for human oversight?",
    "What are the exact fine amounts for high-risk AI systems?",  # the Phase 7 failure case
]

print("\n====== UPGRADED RAG PIPELINE TEST (SEMANTIC RETRIEVAL) ======\n")
rag_results = []

for query in test_queries:
  print(f"\nQuery: {query}")
  result = rag_query(query)
  print(f"Sources used: {result['sources']}")
  print(f"Answer: {result['answer'][:200]}...")
  print(f"Grounded: {result['grounded']}")
  print()
  rag_results.append(result)
  time.sleep(45)

print(f"Total queries: {len(rag_results)}")
print(f"Grounded responses: {sum(1 for r in rag_results if r['grounded'])}")
print("\nUpgraded RAG pipeline ready ✅")

# ============== SAVE RESULTS TO DRIVE ==============

results_df = pd.DataFrame(rag_results)
results_df.to_csv(SAVE_PATH + "phase9_chroma_rag_results.csv", index=False)
print(f"Saved {len(results_df)} results to {SAVE_PATH}phase9_chroma_rag_results.csv")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemet

/tmp/ipykernel_5738/1631447328.py:87: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  embed_fn = GeminiEmbeddingFunction()


====== CHROMA KNOWLEDGE BASE LOADED ======
   doc_001: EU AI Act Article 5 - Prohibited Practices
   doc_002: EU AI Act Article 10 - Data Governance
   doc_003: EU AI Act Article 14 - Human Oversight
   doc_004: EU AI Act Article 99 - Penalties
   doc_005: NIST AI RMF - Four Core Functions
   doc_006: NIST AI 600-1 - Generative AI Risk

Total documents in collection: 6

Chroma knowledge base ready (semantic retrieval, gemini-embedding-001) ✅

====== UPGRADED RAG PIPELINE TEST (SEMANTIC RETRIEVAL) ======


Query: What does the EU AI Act say about prohibited AI practices?
Sources used: ['EU AI Act 2024/1689, Article 5', 'EU AI Act 2024/1689, Article 99']
Answer: Based on the provided source documents, the EU AI Act says the following about prohibited AI practices:

**Prohibited AI Practices (Article 5):**
The Act prohibits the following specific AI practices:...
Grounded: True


Query: What are the penalty tiers under the EU AI Act?
   Waiting 30s... (attempt 1/3)
   Waiting 60s... (atte

# Phase 9: Lesson 1: Chroma Migration
## Findings and Analysis

**System:** gemini-flash-latest with Chroma vector retrieval (gemini-embedding-001)
**Knowledge base:** 6 documents covering EU AI Act
                    Articles 5, 10, 14, 99 and
                    NIST AI RMF core functions and AI 600-1
**Date:** June 2026
**Regulatory mapping:** NIST AI 600-1 hallucination tracking,
                        source data lineage verification

---

## What Was Built

A complete migration of the Phase 7 RAG pipeline from a
Python dictionary with keyword-overlap scoring to a real
vector database, consisting of three changes:

1. The 6-document knowledge base moved from a Python dict
   (KNOWLEDGE_BASE) into a Chroma collection, with each
   document embedded using gemini-embedding-001 instead of
   stored as raw text scored by keyword frequency.

2. retrieve_documents(), which queries Chroma using cosine similarity      
over semantic embeddings instead of keyword overlap, with separate
   RETRIEVAL_DOCUMENT and RETRIEVAL_QUERY task types for
   documents being stored versus queries being searched.

3. rag_query() was kept functionally identical in its return
   shape (query, answer, sources, titles, grounded, doc_count)
   so the Observer Agent and Phase 8 orchestration require no
   downstream changes.

---

## Chroma Pipeline Test Results

| Query | Sources Retrieved | Grounded |
|---|---|---|
| EU AI Act prohibited practices | Article 5, Article 99 | True |
| EU AI Act penalty tiers | Article 99, Article 5 | True |
| NIST AI RMF four functions | NIST RMF 1.0, NIST AI 600-1 | True |
| Article 14 human oversight | Article 14, Article 10 | True |
| Exact fine amounts for high-risk AI systems | Article 99, Article 10 | True |

All 5 responses grounded: 5 out of 5 (100%)

---

## Key Findings

### Finding 1: The Phase 7 Retrieval Bug Is Fixed
The query "What are the exact fine amounts for high-risk AI
systems?" was the documented failure case from Phase 7
Lesson 1: keyword scoring retrieved Article 5 (prohibited
practices) instead of Article 99 (penalties), because both
mention the phrase "AI systems."

Under semantic retrieval, the same query now correctly
retrieves Article 99 and Article 10, the two articles
actually relevant to high-risk system penalties and data
governance. Article 5 is correctly excluded. This is the
direct, citable fix for the Phase 7 Finding 2 limitation.

### Finding 2: Embeddings Require Task-Type Differentiation
Chroma's embedding function was implemented with a
document_mode flag that switches between RETRIEVAL_DOCUMENT
(when storing the 6 source documents) and RETRIEVAL_QUERY
(when embedding an incoming query). Phase 7 had no equivalent
concept, every piece of text was scored identically regardless
of its role. This task-type distinction is itself part of why
retrieval quality improved, not just the move to embeddings
generally.

### Finding 3: Two Real Code Defects Were Found and Fixed During Migration
The Phase 7 retrieve_ducuments() function name was corrected
to retrieve_documents(). Separately, the original ask_llm()
retry function had a logic defect: the "max retries exceeded"
return statement was reachable after only the first failed
attempt rather than after all retries were exhausted, due to
incorrect indentation. Both defects were fixed during this
migration and verified independently.

### Finding 4: Rate Limiting on Generation, Not Retrieval
During testing, several queries hit 429/503 retries on the
ask_llm() generation call, not on embedding or retrieval.
Chroma's retrieval step requires no LLM call beyond the
one-time embedding of documents and queries, and returned
results instantly in every test run. Increasing the delay
between test queries from 15 to 30 seconds resolved the
generation-side rate limiting. This confirms Phase 7's
Finding 4 (retrieval is free and instant) still holds true
under the Chroma migration.

---

## Model Migration Note

text-embedding-004, the model implied by earlier Phase 9
planning, was deprecated by Google on 14 January 2026 and is
no longer callable. This migration uses gemini-embedding-001,
the current production embedding model, confirmed via current
documentation before implementation.

---

## NIST AI 600-1 Mapping

| Requirement | Implementation | Status |
|---|---|---|
| Hallucination tracking | Grounded prompt restricts answers to retrieved sources | Implemented |
| Source data lineage | Every response includes source citations | Implemented |
| Output monitoring | Retrieval and similarity distance logged per query | Implemented |
| Confabulation prevention | Model instructed to state when documents do not contain the answer | Implemented |

---

## Recommendation
Proceed to Phase 9 Lesson 2: rebuild the Phase 8 multi-agent
orchestration using LangGraph in place of pure Python control
flow. The semantic retrieval layer built in this lesson
becomes the RAG node inside that graph, replacing the Phase 7
keyword-based retrieval call without further modification.